processing results of [post_farneback.ipynb](post_farneback.ipynb)

Mac: `femba` Micromamba env

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json

import numpy as np

data_dir = Path("data/farneback")
video_dir = data_dir / "subj1_cam1" / "demo_merge"
out_path = video_dir / "top_clusters.json"

with out_path.open("r", encoding="utf-8") as f:
    loaded_clusters = json.load(f)

loaded_clusters[0]["times"] = np.asarray(loaded_clusters[0]["times"])

In [ ]:
import mne
from src.video_utils import parse_timestamp_from_filename

eeg_fname = "27-MAY-2025_22h19m28.399s.edf"
_, sec_start_eeg = parse_timestamp_from_filename(eeg_fname)
eeg_path = data_dir / eeg_fname
raw = mne.io.read_raw_edf(str(eeg_path))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from src.utils_spectral_power import alpha_delta_powers, alpha_band, delta_band


def fmt_time(t: float) -> str:
    h = int(t // 3600) % 24
    m = int((t % 3600) // 60)
    s = t % 60
    return f"{h:02d}h{m:02d}m{s:06.3f}s"


half_window_sec = 10 * 60
duration_sec = 2 * half_window_sec

for i_cluster, cluster in enumerate(loaded_clusters):
    peak_center_video_sec = cluster["t_peak"]
    peak_center_eeg_sec = peak_center_video_sec - sec_start_eeg

    times, alpha_power, delta_power, raw_fragment = alpha_delta_powers(raw, duration_sec, peak_center_eeg_sec)
    # Store for later analysis
    cluster["eeg_times"] = times.tolist()
    cluster["alpha_power"] = alpha_power.tolist()
    cluster["delta_power"] = delta_power.tolist()

    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    axes[0].plot(times, alpha_power)
    axes[0].set_ylabel("Alpha power ($V^2$)")
    axes[0].set_title(
        f"Cluster #{i_cluster} | "
        f"Alpha-band power ({alpha_band[0]}–{alpha_band[1]} Hz)  |  "
        # f"t_start={fmt_time(cluster['t_start'])}  "
        # f"t_end={fmt_time(cluster['t_end'])}  "
        f"t_peak={fmt_time(cluster['t_peak'])}"
    )
    axes[0].set_ylim([0, 1e-10])
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(times, delta_power)
    axes[1].set_ylabel("Delta power ($V^2$)")
    axes[1].set_xlabel("Time from start of video-aligned fragment (s)")
    axes[1].set_title(f"Delta-band power ({delta_band[0]}–{delta_band[1]} Hz)")
    axes[1].set_ylim([0, 0.8 * 1e-9])
    axes[1].grid(True, alpha=0.3)

    t_start_eeg = cluster["t_start"] - sec_start_eeg - peak_center_eeg_sec + half_window_sec
    t_end_eeg   = cluster["t_end"] -  sec_start_eeg - peak_center_eeg_sec + half_window_sec

    # Vertical lines marking the peak's start/end on both panels
    for ax in axes:
        ax.axvline(t_start_eeg, color="C2", linestyle="--", linewidth=0.8)
        ax.axvline(t_end_eeg,   color="C3", linestyle="--", linewidth=0.8)

    # Inline labels at the top of the first panel
    y_top = axes[0].get_ylim()[1]
    axes[0].text(
        t_start_eeg, y_top * 0.95, " start moving",
        color="C2", fontsize=7, va="top", ha="right",
    )
    axes[0].text(
        t_end_eeg, y_top * 0.95, "end moving",
        color="C3", fontsize=7, va="top", ha="left",
    )

    plt.tight_layout()
    plt.show()

Average spectral power over selected clusters: for subj1 - 0, 3, 4, 11

In [ ]:
sfreq = raw.info["sfreq"]

In [ ]:
selected_cluster_indices = [0, 3, 4, 11]
selected_clusters = np.array(loaded_clusters)[selected_cluster_indices]
alpha_powers, delta_powers = [], []

for c in selected_clusters:
    alpha_powers.append(c["alpha_power"])
    delta_powers.append(c["delta_power"])

times = np.array(selected_clusters[0]["eeg_times"])

alpha_powers_np = np.array(alpha_powers)
delta_powers_np = np.array(delta_powers)

In [ ]:
def plot_averages(_times: np.array, _alpha_center: np.array, _delta_center: np.array,
                  _alpha_conf_band: np.array, _delta_conf_band: np.array, 
                  n_clusters: int, title: str) -> None:
    _, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    
    # --- Alpha subplot ---
    axes[0].plot(_times, _alpha_center, color="tab:blue")
    axes[0].fill_between(
        _times,
        _alpha_center - _alpha_conf_band,
        _alpha_center + _alpha_conf_band,
        color="tab:blue", alpha=0.25,
    )
    axes[0].set_ylabel("Alpha power ($V^2$)")
    axes[0].set_title(
        f"{title} across {n_clusters} clusters | "
        f"Alpha-band power ({alpha_band[0]}–{alpha_band[1]} Hz)"
    )
    axes[0].set_ylim([0, 1e-10])
    axes[0].grid(True, alpha=0.3)

    # --- Delta subplot ---
    axes[1].plot(_times, _delta_center, color="tab:orange")
    axes[1].fill_between(
        _times,
        _delta_center - _delta_conf_band,
        _delta_center + _delta_conf_band,
        color="tab:orange", alpha=0.25,
    )
    axes[1].set_ylabel("Delta power ($V^2$)")
    axes[1].set_xlabel("Time from start of video-aligned fragment (s)")
    axes[1].set_title(f"Delta-band power ({delta_band[0]}–{delta_band[1]} Hz)")
    axes[1].set_ylim([0, 0.8 * 1e-9])
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Average across rows (clusters) -> shape (n_times,)
alpha_mean = np.mean(alpha_powers_np, axis=0)
delta_mean = np.mean(delta_powers_np, axis=0)

# Standard error of the mean across clusters
alpha_sem = np.std(alpha_powers_np, axis=0, ddof=1) / np.sqrt(alpha_powers_np.shape[0])
delta_sem = np.std(delta_powers_np, axis=0, ddof=1) / np.sqrt(delta_powers_np.shape[0])

plot_averages(times, alpha_mean, delta_mean, alpha_sem, delta_sem, 
              len(selected_clusters), "Mean")


In [ ]:
rng = np.random.default_rng(0)
n_boot = 2000
n_clusters, n_times = alpha_powers_np.shape

boot_alpha = np.empty((n_boot, n_times))
boot_delta = np.empty((n_boot, n_times))
for b in range(n_boot):
    idx = rng.integers(0, n_clusters, n_clusters)
    boot_alpha[b] = np.median(alpha_powers_np[idx], axis=0)
    boot_delta[b] = np.median(delta_powers_np[idx], axis=0)

alpha_lo, alpha_hi = np.percentile(boot_alpha, [2.5, 90.], axis=0)
delta_lo, delta_hi = np.percentile(boot_delta, [2.5, 90.], axis=0)

alpha_med = np.median(alpha_powers_np, axis=0)
delta_med = np.median(delta_powers_np, axis=0)

plot_averages(times, alpha_med, delta_med, alpha_hi - alpha_lo, delta_hi - delta_lo,
              len(selected_clusters), "Median")